In [11]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [13]:
LIN_APPROX = 5

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = False

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [14]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="model11_quad_reform.json", endpoints=False)
val, time, x = solve_lp_return_x()
print(x)
quad, q_time = solve_qcp(f_name="model11_quad_reform.json")

print(condition_number_nes_basic())
print(f"Error: {val - quad}")

[159.9999999999805, 168.2539682178661, 140.0000000000706, 99.99999999993763, 68.25396821787417, 131.74603178197236, 119.99999999953963, 80.00000000001435, 159.9999999999161, 200.00000000007142, 2472.5063217225998, 1764.4115608849502, 2171.5660482086496, 3525.862681393754, 4240.000000004016, 2865.49891344915, 1600.0000000003238, 1817.5186678712673, 3355.919939024593, 3599.9999999990832]
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 20 columns and 0 nonzeros (Min)
Model fingerprint: 0xdd0b2740
Model has 10 linear objective coefficients
Model has 20 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [2e-02, 2e-02]
  QLMatrix range   [1e+00, 2e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [5e+0

In [ ]:
# In this case there are 10 quadratic variables
# Define a set of functions for each

# TODO: It should intelligently search for the quadratic variables, not just assume they're at the start

quads = [[val] for val in x[:10]]
iters = 10

for i in range(iters):
    print(f"Iteration {i}")
    points_functions = []
    

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            refinement_points = len(qs)
            uniform = np.linspace(lower, upper, num - refinement_points)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            return uniform
        return np_uniform_add_qs

    for q in quads:
        points_functions.append(np_uniform_add_q_factory(q))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + 1, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="model11_quad_reform.json")

    # quadratic_lin_approx_no_surrogate_refine(1, OUTER_APPROXIMATION, points_function=points_functions,
    #                                             remove_division=REMOVE_DIVISION,
    #                                             f_name="model11_quad_reform_nofix.json")
    val, time, x = solve_lp_return_x()
    # print(f"x: {x}")
    # print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[j])

print(x)
print(f"Error: {val - quad}")

Iteration 0
Error: 99.27135568468111
Iteration 1
Error: 4.622211712452554
Iteration 2
Error: 0.14130415769886895
Iteration 3
Error: 0.004316169660341984
Iteration 4
Error: 0.00021956061209493782
[159.99999978012806, 146.41015518316215, 139.9999999641131, 100.00000025708927, 46.4101557951619, 153.589842918828, 119.99999958396899, 80.00000108413286, 159.99999978020804, 199.99999946402772, 2868.536152648824, 2361.430433536896, 2310.4759218190356, 3751.4036653052153, 4240.000000000039, 3332.209465742293, 1856.2774821672979, 2124.8623899632075, 3410.7457172918835, 3599.9999975406436]
Error: 0.00021956061209493782
